In [1]:
import pandas as pd
import json
from pathlib import Path

BASE_DIR = Path("/home/darshani/lightkurve-env/space-debris-detector")
RAW = BASE_DIR / "data/raw"
CLEANED = BASE_DIR / "data/cleaned"

CLEANED.mkdir(parents=True, exist_ok=True)

In [2]:
def clean_df(df):

    
    df = df.copy()
    df.columns = [c.strip() for c in df.columns] 
    df = df.drop_duplicates()
    df = df.dropna(axis=1, how="all")
    return df

def fix_dates(df):
    df = df.copy()
    for col in df.columns:
        
        if pd.api.types.is_period_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].astype(str)
        
        elif df[col].dtype == "object":
            try:
                df[col] = df[col].apply(lambda x: str(x).strip())
            except:
                pass
    return df


In [3]:
# TLE — flatten nested stuff so nothing comes as null
path = RAW / "TLE_latestt.json"
print("loading TLE:", path)

with open(path, "r") as f:
    tle_data = json.load(f)

flattened = []
for item in tle_data:
    row = {}
    for k, v in item.items():
        # if the value is dict, flatten it
        if isinstance(v, dict):
            for subk, subv in v.items():
                row[f"{k}_{subk}"] = subv
        else:
            row[k] = v
    flattened.append(row)

tle_df = pd.DataFrame(flattened)

tle_df = clean_df(tle_df)
tle_df = fix_dates(tle_df)

out = CLEANED / "tle_cleaned.parquet"
tle_df.to_parquet(out, index=False, engine="fastparquet")  #for some reason pyarrow

print("yay! saved TLE, no more nulls:", out)


loading TLE: /home/darshani/lightkurve-env/space-debris-detector/data/raw/TLE_latestt.json


/tmp/ipykernel_26121/1029467138.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):


yay! saved TLE, no more nulls: /home/darshani/lightkurve-env/space-debris-detector/data/cleaned/tle_cleaned.parquet


In [4]:
# SATCAT — flatten + clean + parquet
path = RAW / "satcat.json"
print("loading SATCAT:", path)

with open(path, "r") as f:
    satcat_data = json.load(f)

flattened = []
for item in satcat_data:
    row = {}
    for k, v in item.items():
        if isinstance(v, dict):
            for subk, subv in v.items():
                row[f"{k}_{subk}"] = subv
        else:
            row[k] = v
    flattened.append(row)

satcat_df = pd.DataFrame(flattened)

# clean + fix dates
satcat_df = clean_df(satcat_df)
satcat_df = fix_dates(satcat_df)

# save parquet
out = CLEANED / "satcat_cleaned.parquet"
satcat_df.to_parquet(out, index=False, engine="fastparquet")
print("saved SATCAT parquet to:", out)


loading SATCAT: /home/darshani/lightkurve-env/space-debris-detector/data/raw/satcat.json


/tmp/ipykernel_26121/1029467138.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):


saved SATCAT parquet to: /home/darshani/lightkurve-env/space-debris-detector/data/cleaned/satcat_cleaned.parquet


In [9]:
# DISCOS objects — nested, flatten, clean, parquet
path = RAW / "discos_objects.json"
print("loading DISCOS objects:", path)

with open(path, "r") as f:
    discos_data = json.load(f)

flattened = []
for item in discos_data:
    row = {"type": item.get("type"), "id": item.get("id")}
    
    # flatten attributes
    attrs = item.get("attributes", {})
    for k, v in attrs.items():
        row[f"attr_{k}"] = v
    
    # flatten relationships
    rels = item.get("relationships", {})
    for k, v in rels.items():
        row[f"rel_{k}"] = str(v)
    
    flattened.append(row)

disc_df = pd.DataFrame(flattened)

# clean + fix dates
disc_df = clean_df(disc_df)
disc_df = fix_dates(disc_df)

# save parquet
out = CLEANED / "discos_cleaned.parquet"
disc_df.to_parquet(out, index=False, engine="fastparquet")
print("saved DISCOS parquet to:", out)


loading DISCOS objects: /home/darshani/lightkurve-env/space-debris-detector/data/raw/discos_objects.json


/tmp/ipykernel_26121/1029467138.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):


saved DISCOS parquet to: /home/darshani/lightkurve-env/space-debris-detector/data/cleaned/discos_cleaned.parquet


In [5]:
path = RAW / "conjunctions.json"
print("loading Conjunctions:", path)

with open(path, "r") as f:
    conj_data = json.load(f)

# flatten if needed (here usually shallow)
flattened = []
for item in conj_data:
    row = {}
    for k, v in item.items():
        if isinstance(v, dict):
            for subk, subv in v.items():
                row[f"{k}_{subk}"] = subv
        else:
            row[k] = v
    flattened.append(row)

conj_df = pd.DataFrame(flattened)

# clean + fix dates
conj_df = clean_df(conj_df)
conj_df = fix_dates(conj_df)

# save parquet
out = CLEANED / "conjunctions_cleaned.parquet"
conj_df.to_parquet(out, index=False, engine="fastparquet")
print("saved Conjunctions parquet to:", out)


loading Conjunctions: /home/darshani/lightkurve-env/space-debris-detector/data/raw/conjunctions.json
saved Conjunctions parquet to: /home/darshani/lightkurve-env/space-debris-detector/data/cleaned/conjunctions_cleaned.parquet


/tmp/ipykernel_26121/1029467138.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):


In [6]:
path = RAW / "decay.json"
print("loading Decay:", path)

with open(path, "r") as f:
    decay_data = json.load(f)

flattened = []
for item in decay_data:
    row = {}
    for k, v in item.items():
        if isinstance(v, dict):
            for subk, subv in v.items():
                row[f"{k}_{subk}"] = subv
        else:
            row[k] = v
    flattened.append(row)

decay_df = pd.DataFrame(flattened)

# clean + fix dates
decay_df = clean_df(decay_df)
decay_df = fix_dates(decay_df)

# save parquet
out = CLEANED / "decay_cleaned.parquet"
decay_df.to_parquet(out, index=False, engine="fastparquet")
print("saved Decay parquet to:", out)


loading Decay: /home/darshani/lightkurve-env/space-debris-detector/data/raw/decay.json


/tmp/ipykernel_26121/1029467138.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):


saved Decay parquet to: /home/darshani/lightkurve-env/space-debris-detector/data/cleaned/decay_cleaned.parquet


In [8]:
path = RAW / "boxscore.json"
print("loading Boxscore:", path)

with open(path, "r") as f:
    box_data = json.load(f)

flattened = []
for item in box_data:
    row = {}
    for k, v in item.items():
        if isinstance(v, dict):
            for subk, subv in v.items():
                row[f"{k}_{subk}"] = subv
        else:
            row[k] = v
    flattened.append(row)

box_df = pd.DataFrame(flattened)

# clean + fix dates
box_df = clean_df(box_df)
box_df = fix_dates(box_df)

# save parquet
out = CLEANED / "boxscore_cleaned.parquet"
box_df.to_parquet(out, index=False, engine="fastparquet")
print("saved Boxscore parquet to:", out)


loading Boxscore: /home/darshani/lightkurve-env/space-debris-detector/data/raw/boxscore.json
saved Boxscore parquet to: /home/darshani/lightkurve-env/space-debris-detector/data/cleaned/boxscore_cleaned.parquet


/tmp/ipykernel_26121/1029467138.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):
